# Decision Trees: Complexity, Overfitting, and Split Rules

This notebook demonstrates how Decision Trees recursively partition feature space, how to extract and visualize the learned partition rules in production, and how to analyze the overfitting profile by tuning the tree depth limit (`max_depth`).

## 1. Import Dependencies and Generate E-Commerce Session Data

We generate synthetic session data for an e-commerce platform. The target is `purchased` (`1` or `0`) based on:
- `session_duration`: Continuous (minutes).
- `page_views`: Continuous.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split

# Seed for reproducibility
np.random.seed(42)

# Generate 400 session samples
m = 400
session_duration = np.random.uniform(1.0, 20.0, m)
page_views = np.random.uniform(1.0, 15.0, m)

# Define a non-linear decision boundary: purchased if views > 6 and duration > 5, or views > 10
purchased = np.zeros(m, dtype=int)
for i in range(m):
    if (page_views[i] > 6.0 and session_duration[i] > 5.0) or (page_views[i] > 11.0):
        # Introduce 10% label noise to simulate production outlier volatility
        purchased[i] = 1 if np.random.rand() > 0.10 else 0
    else:
        purchased[i] = 1 if np.random.rand() < 0.05 else 0

df = pd.DataFrame({
    'session_duration': session_duration,
    'page_views': page_views,
    'purchased': purchased
})

X = df[['session_duration', 'page_views']]
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Class distribution in training: \n{y_train.value_counts(normalize=True)}")

## 2. Train Decision Tree and Export Learned Split Rules

We train a tree with `max_depth=3` to extract the rules.

In [ ]:
model_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
model_shallow.fit(X_train, y_train)

# Export split rules as text
rules = export_text(model_shallow, feature_names=list(X.columns))
print("--- Decision Tree Split Rules ---")
print(rules)

## 3. Analyze Overfitting Profile vs. Tree Depth (`max_depth`)

Let's sweep tree depths from 1 to 15 and monitor training vs test accuracies to identify the exact onset of overfitting.

In [ ]:
depths = range(1, 16)
train_scores = []
test_scores = []

for depth in depths:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(depths, train_scores, marker='o', label='Train Accuracy', color='#339af0', linewidth=2)
plt.plot(depths, test_scores, marker='s', label='Test Accuracy', color='#e03131', linewidth=2)
plt.axvline(x=4, color='#2b8a3e', linestyle='--', label='Optimal Complexity (max_depth=4)')
plt.title("Decision Tree Depth Complexity Sweep")
plt.xlabel("max_depth")
plt.ylabel("Accuracy Score")
plt.xticks(depths)
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

### Interpretation
- **Underfitting Zone (depth < 3):** Both train and test accuracies are low. The model lacks the capacity to represent the rule.
- **Optimal Zone (depth = 4):** Test accuracy peaks. The model captures the underlying pattern without fitting label noise.
- **Overfitting Zone (depth > 6):** Training accuracy climbs to near 100%, but test accuracy drops significantly. The model is splitting on random label noise outliers.

## 4. Visualizing Decision Boundaries

Let's compare the decision surface of an optimal tree (`max_depth=4`) vs. an overfitted tree (`max_depth=15`).

In [ ]:
model_optimal = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
model_overfit = DecisionTreeClassifier(max_depth=15, random_state=42).fit(X_train, y_train)

# Create grid bounds
xx, yy = np.meshgrid(np.linspace(1, 20, 300), np.linspace(1, 15, 300))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

Z_optimal = model_optimal.predict(grid_pts).reshape(xx.shape)
Z_overfit = model_overfit.predict(grid_pts).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot Optimal Tree Decision Surface
axes[0].contourf(xx, yy, Z_optimal, alpha=0.2, colors=['#adb5bd', '#339af0'])
axes[0].scatter(X_train.iloc[:, 0], X_train.iloc[:, 1], c=y_train, cmap='coolwarm_r', edgecolors='k', s=20, alpha=0.7)
axes[0].set_title("Optimal Decision Surface (max_depth=4)")
axes[0].set_xlabel("Session Duration")
axes[0].set_ylabel("Page Views")

# Plot Overfit Tree Decision Surface
axes[1].contourf(xx, yy, Z_overfit, alpha=0.2, colors=['#adb5bd', '#339af0'])
axes[1].scatter(X_train.iloc[:, 0], X_train.iloc[:, 1], c=y_train, cmap='coolwarm_r', edgecolors='k', s=20, alpha=0.7)
axes[1].set_title("Overfit Decision Surface (max_depth=15)")
axes[1].set_xlabel("Session Duration")
axes[1].set_ylabel("Page Views")

plt.tight_layout()
plt.show()

### Conclusion and Tips
1. **Analyze Rule Complexity:** Always run `export_text` or inspect `model.get_depth()` on a newly trained tree. If depth is $>8$, it is likely memorizing noise.
2. **Regularization Constraints:** Always tune `min_samples_leaf` (e.g., set to $\ge 5$ or $1\%$ of dataset size) along with `max_depth`. This prevents splits that isolate individual noise samples.